In [1]:
import urllib.request

In [2]:
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
urllib.request.urlretrieve(url, 'input.txt')

('input.txt', <http.client.HTTPMessage at 0x1db0c373fe0>)

In [3]:
with open('input.txt', 'r', encoding = 'UTF-8') as f:
    text = f.read()

In [4]:
print('Length of dataset: ', len(text))
print(text[:1000])

Length of dataset:  1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, 

In [5]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [6]:
stoi = { ch:i for i, ch in enumerate(chars)}
itos = { i:ch for i, ch in enumerate(chars)}
encode = lambda s:[stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])
print(encode('hi guys'))
print(decode([46, 47]))


[46, 47, 1, 45, 59, 63, 57]
hi


In [7]:
import torch
data = torch.tensor(encode(text), dtype = torch.long)
print(data.shape, data.dtype)
print(data[:100])

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])


In [8]:
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

In [9]:
block_size = 8
train_data[:block_size+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [10]:
x = train_data[:block_size]
y = train_data[1: block_size + 1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f'whet input is {context}, output is {target}')

whet input is tensor([18]), output is 47
whet input is tensor([18, 47]), output is 56
whet input is tensor([18, 47, 56]), output is 57
whet input is tensor([18, 47, 56, 57]), output is 58
whet input is tensor([18, 47, 56, 57, 58]), output is 1
whet input is tensor([18, 47, 56, 57, 58,  1]), output is 15
whet input is tensor([18, 47, 56, 57, 58,  1, 15]), output is 47
whet input is tensor([18, 47, 56, 57, 58,  1, 15, 47]), output is 58


In [11]:
torch.manual_seed(42)
batch_size = 4
block_size = 8

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i + block_size] for i in ix])
    y = torch.stack([data[i+1: i +block_size + 1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print('--------')

for b in range(batch_size):
    for t in range(block_size):
        context = xb[b, :t+1]
        target = yb[b, t]
        print(f'when input is {context.tolist()} the target : {target}')

inputs:
torch.Size([4, 8])
tensor([[57,  1, 46, 47, 57,  1, 50, 53],
        [ 1, 58, 46, 43, 56, 43,  1, 41],
        [17, 26, 15, 17, 10,  0, 32, 53],
        [57, 58,  6,  1, 61, 47, 58, 46]])
targets:
torch.Size([4, 8])
tensor([[ 1, 46, 47, 57,  1, 50, 53, 60],
        [58, 46, 43, 56, 43,  1, 41, 39],
        [26, 15, 17, 10,  0, 32, 53,  1],
        [58,  6,  1, 61, 47, 58, 46,  0]])
--------
when input is [57] the target : 1
when input is [57, 1] the target : 46
when input is [57, 1, 46] the target : 47
when input is [57, 1, 46, 47] the target : 57
when input is [57, 1, 46, 47, 57] the target : 1
when input is [57, 1, 46, 47, 57, 1] the target : 50
when input is [57, 1, 46, 47, 57, 1, 50] the target : 53
when input is [57, 1, 46, 47, 57, 1, 50, 53] the target : 60
when input is [1] the target : 58
when input is [1, 58] the target : 46
when input is [1, 58, 46] the target : 43
when input is [1, 58, 46, 43] the target : 56
when input is [1, 58, 46, 43, 56] the target : 43
when inp

In [12]:
print(xb)

tensor([[57,  1, 46, 47, 57,  1, 50, 53],
        [ 1, 58, 46, 43, 56, 43,  1, 41],
        [17, 26, 15, 17, 10,  0, 32, 53],
        [57, 58,  6,  1, 61, 47, 58, 46]])


In [13]:
import torch 
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(42)

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets = None):
        logits = self.token_embedding_table(idx) # (Batch, Tokens, Channels)

        if targets is None:
            loss = None
        else:    
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B*T)    
            loss = F.cross_entropy(logits, targets)
        return logits, loss


    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, loss = self((idx))
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim = -1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim = 1)
        return idx

m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape) 
print(loss)       
idx = torch.zeros((1, 1), dtype=torch.long)
print(decode(m.generate(idx , max_new_tokens=100)[0].tolist()))

torch.Size([32, 65])
tensor(4.8865, grad_fn=<NllLossBackward0>)

o$,q&IWqW&xtCjaB?ij&bYRGkF?b; f ,CbwhtERCIfuWr,DzJERjhLlVaF&EjffPHDFcNoGIG'&$qXisWTkJPw
 ,b Xgx?D3sj


In [14]:
optimizer = torch.optim.AdamW(m.parameters(), lr = 1e-3 )

In [15]:
batch_size = 32
for steps in range(1000):
    xb, yb = get_batch('train')
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())    

3.764620542526245


In [16]:
print(decode(m.generate(idx = torch.zeros((1, 1), dtype = torch.long), max_new_tokens = 300)[0].tolist()))


Eu;:G,ul.OPA:Hu lwPczaqmKpc'w$nISlsnaq!xt: ghSLq'dqaF:m;hSOZNn'rg$ks3IMJ.JWdLRLJqG'cULgryQHbNLu'V,gro,l.TwVdTG;BrquY?;JSel y;LnimWsKp?,MERZPn;
Qag.cQqjCat,FS$,- trASAhSFOnGTZDAcSkiyZicoHuolHEDso
gonffZi:JCpMpGVflMdVYqy;wcEnfffpdV! QKp?FRseIAFczBuXDNTf
gst-M,nffBQjY3HRfCscecI!,utN rBicYTNwjYw,En'i.k!


In [17]:
torch.manual_seed(42)
B, T, C = 4, 8 ,2
x = torch.randn(B, T, C)
x.shape

torch.Size([4, 8, 2])

In [18]:
#version 1
xbow = torch.zeros((B, T, C))
for b in range(B):
    for t in range(T):
        xprev = x[b, :t+1]
        xbow[b,t] = torch.mean(xprev, 0)

In [19]:
#version 2
wei = torch.tril(torch.ones(T,T))
wei = wei/ wei.sum(1, keepdim=True)
xbow2 = wei @ x # (B, T, T) @ *(B, T, C) ----> (B, T, C)
torch.allclose(xbow, xbow2)

True

In [20]:
#version 3
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T, T))
wei= wei.masked_fill(tril ==0, float('-inf'))
wei = F.softmax(wei, dim = 1)
xbow3 = wei @ x
torch.allclose(xbow, xbow3)

True

In [ ]:
#version 4 Self-Attention Mechanism
torch.manual_seed(42)
B, T, C = 4, 8, 32
x = torch.randn(B, T, C)

head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias = False)
value = nn.Linear(C, head_size, bias = False)
k = key(x) #(B, T, 16)
q = query(x) # (B, T, 16)
wei = q @ k.transpose(-2, -1) # (B, T, 16) @ (B, 16, T  ) ----> (B, T, T)

tril = torch.tril(torch.ones(T, T))
#wei = torch.zeros((T, T))
wei = wei.masked_fill(tril ==0, float('-inf'))
wei = F.softmax(wei, dim = 1)

v= value(x)
out = wei @ v

out.shape

torch.Size([4, 8, 16])